<a href="https://colab.research.google.com/github/officialselun-design/pysr/blob/master/Sel%C3%BBn_AI_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install praat-parselmouth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 90.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import scipy.linalg as la
import scipy.integrate as integrate
import scipy.interpolate as interp
import scipy.signal as signal
import warnings
warnings.filterwarnings('ignore')
class NumericalMathematicsEngine:
    def __init__(self, eps: float = np.finfo(np.float64).eps):
        self.eps = eps
    def matrix_diagnostics(self, A: np.ndarray) -> dict:
        cond_2 = float(np.linalg.cond(A, 2))
        cond_fro = float(np.linalg.cond(A, 'fro'))
        rank = int(np.linalg.matrix_rank(A, tol=self.eps * max(A.shape) * np.max(np.abs(A))))
        U, S, Vh = la.svd(A)
        tol = self.eps * max(A.shape) * np.max(S)
        pinv_A = la.pinv(A, atol=tol)
        return {"Condition_Number_2Norm": cond_2, "Condition_Number_Frobenius": cond_fro, "Matrix_Rank": rank, "Singular_Values": S.tolist(), "PseudoInverse_Norm": float(np.linalg.norm(pinv_A, 2))}
    def stable_cholesky(self, A: np.ndarray, jitter: float = 1e-10) -> np.ndarray:
        try:
            return la.cholesky(A, lower=True)
        except la.LinAlgError:
            return la.cholesky(A + jitter * np.eye(A.shape[0]), lower=True)
    def kahan_sum(self, arr: np.ndarray) -> float:
        sum_val, c = 0.0, 0.0
        for x in arr.flat:
            y = x - c
            t = sum_val + y
            c = (t - sum_val) - y
            sum_val = t
        return float(sum_val)
    def log_sum_exp(self, a: np.ndarray) -> float:
        max_a = np.max(a)
        return float(max_a + np.log(np.sum(np.exp(a - max_a))))
    def complex_step_derivative(self, func, x: float, h: float = 1e-20) -> float:
        return float(np.imag(func(x + 1j * h)) / h)
    def adaptive_central_difference(self, func, x: float) -> dict:
        h = np.sqrt(self.eps) * max(abs(x), 1.0)
        d1 = (func(x + h) - func(x - h)) / (2.0 * h)
        d2 = (func(x + h) - 2.0 * func(x) + func(x - h)) / (h ** 2)
        return {"Derivative": float(d1), "Second_Derivative": float(d2), "Optimal_Step_Size": float(h), "Estimated_Truncation_Error": float(np.abs((h**2 / 6.0) * d2))}
    def adaptive_quadrature(self, func, a: float, b: float, tol: float = 1e-12) -> dict:
        val, err = integrate.quad(func, a, b, epsabs=tol, epsrel=tol)
        return {"Integral_Value": float(val), "Absolute_Error_Bound": float(err)}
    def simpson_composite(self, y: np.ndarray, dx: float = 1.0) -> dict:
        return {"Simpson_Integral": float(integrate.simpson(y, dx=dx))}
    def barycentric_lagrange_interp(self, x_nodes: np.ndarray, y_nodes: np.ndarray, x_eval: np.ndarray) -> np.ndarray:
        return interp.BarycentricInterpolator(x_nodes, y_nodes)(x_eval)
    def cubic_spline_interp(self, x_nodes: np.ndarray, y_nodes: np.ndarray, x_eval: np.ndarray) -> np.ndarray:
        return interp.CubicSpline(x_nodes, y_nodes, bc_type='natural')(x_eval)
    def polyphase_resampling(self, x: np.ndarray, up: int, down: int) -> np.ndarray:
        return signal.resample_poly(x, up, down)
    def fourier_sinc_resampling(self, x: np.ndarray, num_samples: int) -> np.ndarray:
        return signal.resample(x, num_samples)
if __name__ == "__main__":
    engine = NumericalMathematicsEngine()
    test_matrix = np.array([[4.0, 12.0, -16.0], [12.0, 37.0, -43.0], [-16.0, -43.0, 98.0]])
    diag_res = engine.matrix_diagnostics(test_matrix)
    chol_res = engine.stable_cholesky(test_matrix)
    test_func = lambda x: np.sin(x)
    comp_diff = engine.complex_step_derivative(test_func, np.pi / 4.0)
    adapt_diff = engine.adaptive_central_difference(test_func, np.pi / 4.0)
    quad_res = engine.adaptive_quadrature(test_func, 0, np.pi)
    tiny_numbers = np.full(100000, 1e-16)
    kahan_res = engine.kahan_sum(tiny_numbers)
    x_nodes = np.linspace(0, 10, 11)
    y_nodes = np.sin(x_nodes)
    x_eval = np.linspace(0, 10, 100)
    bary_interp = engine.barycentric_lagrange_interp(x_nodes, y_nodes, x_eval)
    resampled_signal = engine.polyphase_resampling(y_nodes, 2, 1)
    print("[LINEAR ALGEBRA DIAGNOSTICS]")
    for k, v in diag_res.items(): print(f"{k}: {v}")
    print("\n[NUMERICAL DIFFERENTIATION]")
    print(f"Complex_Step_Derivative (Exact): {comp_diff:.15f}")
    print(f"Adaptive_Central_Diff: {adapt_diff['Derivative']:.15f}")
    print("\n[NUMERICAL INTEGRATION]")
    print(f"Adaptive_Quad_Exact_Pi: {quad_res['Integral_Value']:.15f}")
    print("\n[FLOATING-POINT ERROR CONTROL]")
    print(f"Kahan_Sum_Result: {kahan_res}")
    print("\n[RESAMPLING & INTERPOLATION]")
    print(f"Barycentric_Interp_Shape: {bary_interp.shape}")
    print(f"Polyphase_Resampled_Length: {len(resampled_signal)}")
import numpy as np
import scipy.signal as signal
import scipy.fft as fft
import warnings
warnings.filterwarnings('ignore')
class SignalMathematicsEngine:
    def __init__(self, fs: int = 16000, eps: float = np.finfo(np.float64).eps):
        self.fs = fs
        self.eps = eps
    def compute_fourier_transform(self, x: np.ndarray) -> dict:
        X = fft.rfft(x)
        return {"Frequencies": fft.rfftfreq(len(x), 1.0 / self.fs), "Magnitude": np.abs(X), "Phase": np.unwrap(np.angle(X)), "Power": (np.abs(X)**2) / len(x)}
    def compute_stft(self, x: np.ndarray, nperseg: int = 512, noverlap: int = 256) -> dict:
        f, t, Zxx = signal.stft(x, fs=self.fs, nperseg=nperseg, noverlap=noverlap)
        return {"Frequencies": f, "Time": t, "Complex_Matrix": Zxx, "Spectrogram": np.abs(Zxx)**2}
    def compute_window_functions(self, win_type: str, N: int) -> dict:
        win = signal.get_window(win_type, N)
        return {"Window": win, "Coherent_Gain": float(np.sum(win) / N), "Processing_Gain": float(np.sum(win**2) / N), "Equivalent_Noise_BW": float(N * np.sum(win**2) / (np.sum(win)**2))}
    def compute_welch_estimation(self, x: np.ndarray, nperseg: int = 512) -> dict:
        f, psd = signal.welch(x, fs=self.fs, nperseg=nperseg)
        return {"Frequencies": f, "PSD": psd, "Peak_Frequency": float(f[np.argmax(psd)])}
    def compute_wavelet_transform(self, x: np.ndarray, widths: np.ndarray = np.arange(1, 31)) -> dict:
        cwt_mat = signal.cwt(x, signal.ricker, widths)
        return {"CWT_Matrix": cwt_mat, "Scalogram": np.abs(cwt_mat)**2, "Total_Energy": float(np.sum(np.abs(cwt_mat)**2))}
    def compute_hilbert_transform(self, x: np.ndarray) -> dict:
        analytic = signal.hilbert(x)
        inst_phase = np.unwrap(np.angle(analytic))
        return {"Analytic_Signal": analytic, "Envelope": np.abs(analytic), "Instantaneous_Phase": inst_phase, "Instantaneous_Frequency": np.diff(inst_phase) / (2.0 * np.pi) * self.fs}
    def compute_cepstrum(self, x: np.ndarray) -> dict:
        X = fft.fft(x)
        log_mag = np.log(np.abs(X) + self.eps)
        return {"Real_Cepstrum": np.real(fft.ifft(log_mag)), "Complex_Cepstrum": np.real(fft.ifft(np.log(X + self.eps))), "Quefrency": np.arange(len(x)) / float(self.fs)}
    def compute_spectral_envelope(self, x: np.ndarray, lifter_cutoff: int = 20) -> dict:
        cepstrum = np.real(fft.ifft(np.log(np.abs(fft.fft(x)) + self.eps)))
        lifter = np.zeros_like(cepstrum)
        lifter[:lifter_cutoff] = 1.0
        lifter[-lifter_cutoff+1:] = 1.0
        return {"Spectral_Envelope": np.exp(np.real(fft.fft(cepstrum * lifter)))[:len(x)//2 + 1]}
    def compute_harmonic_analysis(self, x: np.ndarray, num_harmonics: int = 5) -> dict:
        autocorr = signal.correlate(x, x, mode='full')[len(x)-1:]
        f0_idx = np.where(np.diff(autocorr) > 0)[0][0] + np.argmax(autocorr[np.where(np.diff(autocorr) > 0)[0][0]:]) if len(np.where(np.diff(autocorr) > 0)[0]) > 0 else 1
        f0 = float(self.fs / f0_idx) if f0_idx > 0 else 0.0
        mag = np.abs(fft.rfft(x))
        freqs = fft.rfftfreq(len(x), 1.0 / self.fs)
        harmonic_powers = [float(mag[np.argmin(np.abs(freqs - (h * f0)))]**2) for h in range(1, num_harmonics + 1)]
        harm_power = float(np.sum(harmonic_powers))
        total_power = float(np.sum(mag**2))
        return {"F0_Estimate": f0, "Harmonic_Powers": harmonic_powers, "THD": float(np.sqrt(np.sum(harmonic_powers[1:])) / (np.sqrt(harmonic_powers[0]) + self.eps)) if len(harmonic_powers) > 1 else 0.0, "HNR_dB": float(10 * np.log10(harm_power / (total_power - harm_power + self.eps)))}
    def compute_cross_correlation(self, x: np.ndarray, y: np.ndarray) -> dict:
        corr = signal.correlate(x - np.mean(x), y - np.mean(y), mode='full')
        norm_corr = corr / (np.std(x) * np.std(y) * len(x) + self.eps)
        lags = signal.correlation_lags(len(x), len(y), mode='full')
        return {"Normalized_Cross_Correlation": norm_corr, "Lags": lags, "Peak_Lag_Samples": int(lags[np.argmax(np.abs(norm_corr))])}
    def compute_auto_correlation(self, x: np.ndarray) -> dict:
        norm_x = x - np.mean(x)
        autocorr = signal.correlate(norm_x, norm_x, mode='full')[len(x)-1:]
        return {"Normalized_Auto_Correlation": autocorr / (autocorr[0] + self.eps), "Positive_Lags": signal.correlation_lags(len(x), len(x), mode='full')[len(x)-1:]}
    def compute_convolution(self, x: np.ndarray, h: np.ndarray, mode: str = 'full') -> dict:
        return {"Linear_Convolution": signal.convolve(x, h, mode=mode), "FFT_Convolution": signal.fftconvolve(x, h, mode=mode)}
    def compute_wiener_deconvolution(self, y: np.ndarray, h: np.ndarray, snr_db: float = 30.0) -> dict:
        N = max(len(y), len(h))
        H = fft.fft(h, N)
        wiener_filter = np.conj(H) / (np.abs(H)**2 + (1.0 / (10.0 ** (snr_db / 10.0))))
        return {"Estimated_Deconvolved_Signal": np.real(fft.ifft(fft.fft(y, N) * wiener_filter)), "Wiener_Filter": wiener_filter}
import numpy as np
import scipy.signal as signal
import scipy.stats as stats
import scipy.fft as fft
import warnings
warnings.filterwarnings('ignore')

class TimeSeriesMathematicsEngine:
    def __init__(self, fs: int = 16000, eps: float = np.finfo(np.float64).eps):
        self.fs = fs
        self.eps = eps

    def compute_time_index(self, num_samples: int, t_start: float = 0.0) -> dict:
        dt = 1.0 / self.fs
        t = t_start + np.arange(num_samples) * dt
        return {"Time_Vector": t, "Sampling_Interval": dt, "Total_Duration": float(num_samples * dt)}

    def compute_frame_index(self, num_samples: int, frame_size: int, hop_size: int) -> dict:
        num_frames = max(0, (num_samples - frame_size) // hop_size + 1)
        frame_indices = np.arange(num_frames)
        start_samples = frame_indices * hop_size
        end_samples = start_samples + frame_size
        start_times = start_samples / float(self.fs)
        center_times = (start_samples + (frame_size / 2.0)) / float(self.fs)
        return {"Num_Frames": int(num_frames), "Start_Samples": start_samples, "End_Samples": end_samples, "Start_Times": start_times, "Center_Times": center_times}

    def compute_time_synchronization(self, x: np.ndarray, y: np.ndarray, method: str = 'gcc_phat') -> dict:
        N = len(x) + len(y) - 1
        X = fft.rfft(x, n=N)
        Y = fft.rfft(y, n=N)
        R = X * np.conj(Y)
        if method == 'gcc_phat':
            R /= (np.abs(R) + self.eps)
        cc = fft.irfft(R, n=N)
        max_idx = np.argmax(np.abs(cc))
        shift = max_idx if max_idx < N // 2 else max_idx - N
        return {"Sample_Offset": int(shift), "Time_Delay_Sec": float(shift / self.fs), "Cross_Correlation": cc}

    def compute_signal_segmentation(self, x: np.ndarray, frame_size: int, hop_size: int, energy_threshold_ratio: float = 0.1) -> dict:
        windows = np.lib.stride_tricks.sliding_window_view(x, frame_size)[::hop_size]
        energies = np.sum(windows**2, axis=-1)
        threshold = energy_threshold_ratio * np.max(energies)
        active_mask = energies > threshold
        active_frames = np.where(active_mask)[0]
        segments = []
        if len(active_frames) > 0:
            split_points = np.where(np.diff(active_frames) > 1)[0] + 1
            groupings = np.split(active_frames, split_points)
            for g in groupings:
                start_samp = int(g[0] * hop_size)
                end_samp = int(min(len(x), g[-1] * hop_size + frame_size))
                segments.append((start_samp, end_samp))
        return {"Frame_Energies": energies, "Active_Mask": active_mask, "Segments_Sample_Bounds": segments}

    def compute_sliding_windows(self, x: np.ndarray, window_length: int, hop_size: int, window_type: str = 'hann') -> dict:
        strided = np.lib.stride_tricks.sliding_window_view(x, window_length)[::hop_size]
        if window_type:
            win = signal.get_window(window_type, window_length)
            windowed_strided = strided * win
        else:
            windowed_strided = strided
        return {"Strided_Matrix": windowed_strided, "Num_Windows": strided.shape[0], "Window_Length": window_length}

    def compute_change_points(self, x: np.ndarray, threshold: float = 5.0) -> dict:
        s = x - np.mean(x)
        cusum_pos = np.zeros(len(x))
        cusum_neg = np.zeros(len(x))
        change_points = []
        for i in range(1, len(x)):
            cusum_pos[i] = max(0, cusum_pos[i-1] + s[i])
            cusum_neg[i] = min(0, cusum_neg[i-1] + s[i])
            if cusum_pos[i] > threshold or abs(cusum_neg[i]) > threshold:
                change_points.append(i)
        return {"CUSUM_Positive": cusum_pos, "CUSUM_Negative": cusum_neg, "Change_Point_Indices": change_points}

    def compute_trend_detection(self, x: np.ndarray) -> dict:
        t = np.arange(len(x)) / float(self.fs)
        slope, intercept, r_value, p_value, std_err = stats.linregress(t, x)
        n = len(x)
        s_stat = np.sum([np.sum(np.sign(x[k + 1:] - x[k])) for k in range(n - 1)])
        _, counts = np.unique(x, return_counts=True)
        var_s = (n * (n - 1) * (2 * n + 5) - np.sum(counts * (counts - 1) * (2 * counts + 5))) / 18.0
        z = (s_stat - np.sign(s_stat)) / np.sqrt(var_s + self.eps) if s_stat != 0 else 0.0
        mk_p_val = 2.0 * (1.0 - stats.norm.cdf(abs(z)))
        return {"Linear_Slope": float(slope), "Linear_Intercept": float(intercept), "R_Squared": float(r_value**2), "Linear_P_Value": float(p_value), "Mann_Kendall_Stat": float(s_stat), "Mann_Kendall_Z": float(z), "Mann_Kendall_P_Value": float(mk_p_val)}

    def compute_drift_estimation(self, x: np.ndarray, reference_freq: float = 1000.0) -> dict:
        t = np.arange(len(x)) / float(self.fs)
        analytic = signal.hilbert(x)
        inst_phase = np.unwrap(np.angle(analytic))
        expected_phase = 2.0 * np.pi * reference_freq * t
        phase_error = inst_phase - expected_phase
        drift_slope = float(np.polyfit(t, phase_error, 1)[0])
        drift_ppm = float((drift_slope / (2.0 * np.pi * reference_freq)) * 1e6)
        return {"Phase_Error": phase_error, "Drift_Rate_Rad_Sec": drift_slope, "Drift_PPM": drift_ppm}
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any

@dataclass
class AudioObject:
    samples: np.ndarray
    sample_rate: int
    duration_sec: float
    num_channels: int = 1
    bit_depth: int = 16
    is_normalized: bool = False
    device_metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class FeatureObject:
    spectral_centroid: np.ndarray
    spectral_bandwidth: np.ndarray
    spectral_rolloff: np.ndarray
    zero_crossing_rate: np.ndarray
    rms_energy: np.ndarray
    mfcc_matrix: np.ndarray
    chroma_matrix: np.ndarray
    delta_mfcc: Optional[np.ndarray] = None
    delta_delta_mfcc: Optional[np.ndarray] = None
    feature_dimensions: Tuple[int, ...] = field(default_factory=tuple)

@dataclass
class HarmonicObject:
    f0_contour: np.ndarray
    harmonic_frequencies_matrix: np.ndarray
    harmonic_amplitudes_matrix: np.ndarray
    thd_contour: np.ndarray
    hnr_contour: np.ndarray
    inharmonicity_coefficient: float
    phase_coherence_matrix: Optional[np.ndarray] = None
    periodicity_mask: Optional[np.ndarray] = None

@dataclass
class TimeSeriesObject:
    time_index: np.ndarray
    frame_centers: np.ndarray
    change_point_indices: List[int]
    active_segments_bounds: List[Tuple[int, int]]
    drift_rate_ppm: float
    trend_slope: float
    trend_intercept: float
    mann_kendall_z: float

@dataclass
class StatisticalObject:
    mean_val: float
    variance_val: float
    skewness_val: float
    kurtosis_val: float
    median_val: float
    interquartile_range: float
    shannon_entropy: float
    percentiles: Dict[str, float]
    covariance_matrix: Optional[np.ndarray] = None
    probability_density: Optional[np.ndarray] = None

@dataclass
class ValidationObject:
    is_valid: bool
    snr_db: float
    clipping_ratio: float
    nan_count: int
    inf_count: int
    anomalies_detected: int
    confidence_score: float
    error_flags: List[str] = field(default_factory=list)
    warning_flags: List[str] = field(default_factory=list)

@dataclass
class ReportObject:
    analysis_id: str
    timestamp_utc: float
    processing_time_ms: float
    audio_meta: AudioObject
    features: FeatureObject
    harmonics: HarmonicObject
    time_series: TimeSeriesObject
    statistics: StatisticalObject
    validation: ValidationObject
    engine_version: str = "SelunCore-Pro-1.0"
    custom_annotations: Dict[str, Any] = field(default_factory=dict)
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional

@dataclass
class GlobalConfig:
    environment: str = "production"
    debug_mode: bool = False
    random_seed: int = 42
    execution_device: str = "cpu"
    max_threads: int = 8
    float_precision: str = "float64"
    eps: float = np.finfo(np.float64).eps

@dataclass
class PhysicalConstants:
    speed_of_sound: float = 343.2
    air_density: float = 1.204
    reference_pressure: float = 2.0e-5
    reference_intensity: float = 1.0e-12
    standard_pitch_a4: float = 440.0
    ambient_temperature_c: float = 20.0
    relative_humidity_pct: float = 50.0

@dataclass
class SignalParams:
    fs: int = 16000
    bit_depth: int = 16
    channels: int = 1
    dc_offset_removal: bool = True
    pre_emphasis_coeff: float = 0.97
    normalize_audio: bool = True
    target_peak_db: float = -1.0
    min_snr_threshold_db: float = 6.0

@dataclass
class AnalysisParams:
    window_length: int = 512
    hop_size: int = 256
    fft_size: int = 1024
    window_type: str = "blackmanharris"
    lifter_cutoff: int = 20
    cwt_width_max: int = 30
    cwt_wavelet_type: str = "ricker"
    vadd_energy_threshold_ratio: float = 0.05
    f0_min_hz: float = 50.0
    f0_max_hz: float = 1000.0
    num_harmonics: int = 5

@dataclass
class StatisticalParams:
    alpha: float = 0.05
    confidence_level: float = 0.95
    n_bootstraps: int = 1000
    mcmc_samples: int = 5000
    fdr_method: str = "fdr_bh"
    cusum_threshold: float = 5.0
    outlier_std_factor: float = 3.0

@dataclass
class UserConfig:
    user_id: str = "researcher_01"
    experiment_id: str = "exp_vocal_acoustics_001"
    export_formats: List[str] = field(default_factory=lambda: ["json", "csv", "npz"])
    save_spectrograms: bool = True
    custom_parameters: Dict[str, Any] = field(default_factory=dict)

@dataclass
class SelunCoreConfig:
    global_cfg: GlobalConfig = field(default_factory=GlobalConfig)
    physics: PhysicalConstants = field(default_factory=PhysicalConstants)
    signal: SignalParams = field(default_factory=SignalParams)
    analysis: AnalysisParams = field(default_factory=AnalysisParams)
    stats: StatisticalParams = field(default_factory=StatisticalParams)
    user: UserConfig = field(default_factory=UserConfig)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "global": self.global_cfg.__dict__,
            "physics": self.physics.__dict__,
            "signal": self.signal.__dict__,
            "analysis": self.analysis.__dict__,
            "stats": self.stats.__dict__,
            "user": self.user.__dict__
        }
        import os
import sys
import gc
import json
import logging
import hashlib
import time
from pathlib import Path
from typing import Any, Dict, Optional, Tuple, Callable, Union
import numpy as np

class SelunCoreError(Exception):
    """Base exception for all Selûn Core system errors."""
    pass

class SignalProcessingError(SelunCoreError):
    """Raised when signal processing fails or inputs are malformed."""
    pass

class MathematicalError(SelunCoreError):
    """Raised on numerical instability, matrix singularity, or convergence failure."""
    pass

class MemoryExceededError(SelunCoreError):
    """Raised when memory consumption exceeds predefined limits."""
    pass

class CacheError(SelunCoreError):
    """Raised when cache retrieval or insertion fails."""
    pass

class FileSystemError(SelunCoreError):
    """Raised on file I/O errors or invalid paths."""
    pass

class ExceptionHandler:
    @staticmethod
    def wrap_safely(default_return: Any = None):
        def decorator(func: Callable):
            def wrapper(*args, **kwargs):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    logging.error(f"Error in {func.__name__}: {str(e)}", exc_info=True)
                    if isinstance(e, SelunCoreError):
                        raise e
                    raise SelunCoreError(f"Unhandled system error during {func.__name__}") from e
            return wrapper
        return decorator

class SystemLogger:
    def __init__(self, name: str = "SelunCore", log_file: Optional[str] = None, level: int = logging.INFO):
        self.logger = logging.getLogger(name)
        self.logger.setLevel(level)
        self.logger.handlers.clear()

        formatter = logging.Formatter(
            fmt="[%(asctime)s.%(msecs)03d] [%(levelname)s] [%(name)s]: %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )

        console_handler = logging.StreamHandler(sys.stdout)
        console_handler.setFormatter(formatter)
        self.logger.addHandler(console_handler)

        if log_file:
            file_handler = logging.FileHandler(log_file)
            file_handler.setFormatter(formatter)
            self.logger.addHandler(file_handler)

    def get_logger(self) -> logging.Logger:
        return self.logger

class MemoryManager:
    def __init__(self, memory_limit_mb: float = 8192.0):
        self.memory_limit_mb = memory_limit_mb

    @staticmethod
    def force_garbage_collection() -> int:
        return gc.collect()

    @staticmethod
    def get_array_memory_mb(arr: np.ndarray) -> float:
        return float(arr.nbytes / (1024.0 * 1024.0))

    def check_memory_bounds(self, estimated_addition_mb: float = 0.0) -> Dict[str, Any]:
        allocated_bytes = sys.getsizeof(gc.get_objects())
        total_mb = float(allocated_bytes / (1024.0 * 1024.0)) + estimated_addition_mb
        exceeds = total_mb > self.memory_limit_mb
        if exceeds:
            self.force_garbage_collection()
        return {"Current_Allocated_MB": total_mb, "Limit_MB": self.memory_limit_mb, "Limit_Exceeded": exceeds}

class ArrayCacheManager:
    def __init__(self, max_capacity: int = 128):
        self.max_capacity = max_capacity
        self._cache: Dict[str, Tuple[np.ndarray, float]] = {}

    @staticmethod
    def generate_hash(arr: np.ndarray) -> str:
        return hashlib.sha256(arr.tobytes()).hexdigest()

    def put(self, key: str, data: np.ndarray) -> None:
        if len(self._cache) >= self.max_capacity:
            oldest_key = min(self._cache.keys(), key=lambda k: self._cache[k][1])
            del self._cache[oldest_key]
        self._cache[key] = (data.copy(), time.time())

    def get(self, key: str) -> Optional[np.ndarray]:
        if key in self._cache:
            data, _ = self._cache[key]
            self._cache[key] = (data, time.time())
            return data.copy()
        return None

    def clear(self) -> None:
        self._cache.clear()
        gc.collect()

class FileManager:
    @staticmethod
    def ensure_directory(directory_path: Union[str, Path]) -> Path:
        p = Path(directory_path)
        p.mkdir(parents=True, exist_ok=True)
        return p

    @classmethod
    def save_json(cls, data: Dict[str, Any], file_path: Union[str, Path]) -> None:
        try:
            path = Path(file_path)
            cls.ensure_directory(path.parent)
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
        except Exception as e:
            raise FileSystemError(f"Failed to write JSON file at {file_path}") from e

    @classmethod
    def load_json(cls, file_path: Union[str, Path]) -> Dict[str, Any]:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            raise FileSystemError(f"Failed to read JSON file at {file_path}") from e

    @classmethod
    def save_numpy_data(cls, data_dict: Dict[str, np.ndarray], file_path: Union[str, Path]) -> None:
        try:
            path = Path(file_path)
            cls.ensure_directory(path.parent)
            np.savez_compressed(path, **data_dict)
        except Exception as e:
            raise FileSystemError(f"Failed to save NPZ file at {file_path}") from e

    @classmethod
    def load_numpy_data(cls, file_path: Union[str, Path]) -> Dict[str, np.ndarray]:
        try:
            with np.load(file_path) as loaded:
                return {k: loaded[k] for k in loaded.files}
        except Exception as e:
            raise FileSystemError(f"Failed to load NPZ file at {file_path}") from e

class AcousticUnitConverter:
    @staticmethod
    def lin_to_db(linear_val: Union[float, np.ndarray], eps: float = 1e-12) -> Union[float, np.ndarray]:
        return 20.0 * np.log10(np.maximum(linear_val, eps))

    @staticmethod
    def db_to_lin(db_val: Union[float, np.ndarray]) -> Union[float, np.ndarray]:
        return 10.0 ** (db_val / 20.0)

    @staticmethod
    def pascal_to_dbspl(pressure_pa: Union[float, np.ndarray], p_ref: float = 2.0e-5, eps: float = 1e-12) -> Union[float, np.ndarray]:
        return 20.0 * np.log10(np.maximum(pressure_pa, eps) / p_ref)

    @staticmethod
    def dbspl_to_pascal(dbspl: Union[float, np.ndarray], p_ref: float = 2.0e-5) -> Union[float, np.ndarray]:
        return p_ref * (10.0 ** (dbspl / 20.0))

    @staticmethod
    def hz_to_mel(hz: Union[float, np.ndarray]) -> Union[float, np.ndarray]:
        return 2595.0 * np.log10(1.0 + (hz / 700.0))

    @staticmethod
    def mel_to_hz(mel: Union[float, np.ndarray]) -> Union[float, np.ndarray]:
        return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

    @staticmethod
    def hz_to_bark(hz: Union[float, np.ndarray]) -> Union[float, np.ndarray]:
        return 13.0 * np.arctan(0.00076 * hz) + 3.5 * np.arctan((hz / 7500.0) ** 2)

    @staticmethod
    def hz_to_semitones(hz: Union[float, np.ndarray], f_ref: float = 440.0, eps: float = 1e-12) -> Union[float, np.ndarray]:
        return 12.0 * np.log2(np.maximum(hz, eps) / f_ref)

class VersionManager:
    MAJOR: int = 1
    MINOR: int = 0
    PATCH: int = 0
    RELEASE_STAGE: str = "PRO_PRODUCTION"

    @classmethod
    def get_version_string(cls) -> str:
        return f"{cls.MAJOR}.{cls.MINOR}.{cls.PATCH}-{cls.RELEASE_STAGE}"

    @classmethod
    def get_version_tuple(cls) -> Tuple[int, int, int]:
        return (cls.MAJOR, cls.MINOR, cls.PATCH)

    @classmethod
    def verify_compatibility(cls, required_major: int, required_minor: int) -> bool:
        if required_major != cls.MAJOR:
            return False
        return cls.MINOR >= required_minor



[LINEAR ALGEBRA DIAGNOSTICS]
Condition_Number_2Norm: 6566.198356197774
Condition_Number_Frobenius: 6617.761130619975
Matrix_Rank: 3
Singular_Values: [123.47723179013157, 15.503963229407578, 0.018804980460814517]
PseudoInverse_Norm: 53.177401703967824

[NUMERICAL DIFFERENTIATION]
Complex_Step_Derivative (Exact): 0.707106781186548
Adaptive_Central_Diff: 0.707106780260801

[NUMERICAL INTEGRATION]
Adaptive_Quad_Exact_Pi: 2.000000000000000

[FLOATING-POINT ERROR CONTROL]
Kahan_Sum_Result: 1e-11

[RESAMPLING & INTERPOLATION]
Barycentric_Interp_Shape: (100,)
Polyphase_Resampled_Length: 22


In [26]:
import numpy as np
import scipy.signal as signal
import parselmouth
from typing import Dict, Tuple, Optional, Any
import warnings
warnings.filterwarnings('ignore')
class SignalAcquisitionEngine:
    """
    Research-grade Audio Acquisition and Preprocessing Module.
    Designed to process complex phonation signals while preserving
    nonlinear acoustic phenomena and multi-source oscillator dynamics.
    """
    def __init__(self, target_sr: float = 44100.0, bit_depth: int = 16):
        self.target_sr = target_sr
        self.bit_depth = bit_depth
        self.max_val = 2 ** (self.bit_depth - 1) - 1
    def load_and_validate(self, file_path: str) -> parselmouth.Sound:
        """Loads audio via Parselmouth and forces mono conversion."""
        snd = parselmouth.Sound(file_path)
        # Mono Conversion
        if snd.get_number_of_channels() > 1:
            snd = snd.extract_channel(1) # Research standard: use primary channel
        return snd
    def remove_dc_offset(self, snd: parselmouth.Sound) -> parselmouth.Sound:
        """Removes DC offset to center the waveform around zero."""
        audio_data = snd.values[0]
        mean_val = np.mean(audio_data)
        corrected_data = audio_data - mean_val
        # Reconstruct parselmouth Sound
        return parselmouth.Sound(corrected_data, snd.sampling_frequency)
    def apply_pre_emphasis(self, snd: parselmouth.Sound, alpha: float = 0.97) -> parselmouth.Sound:
        """
        Applies a pre-emphasis filter.
        Note: Alpha can be adjusted or set to 0 to avoid suppressing low-frequency subharmonics.
        """
        audio_data = snd.values[0]
        emphasized = np.append(audio_data[0], audio_data[1:] - alpha * audio_data[:-1])
        return parselmouth.Sound(emphasized, snd.sampling_frequency)
    def normalize_signal(self, snd: parselmouth.Sound, target_peak: float = 0.95) -> parselmouth.Sound:
        """Normalizes audio to a target peak amplitude to prevent clipping."""
        audio_data = snd.values[0]
        max_amp = np.max(np.abs(audio_data))
        if max_amp > 0:
            normalized_data = (audio_data / max_amp) * target_peak
        else:
            normalized_data = audio_data
        return parselmouth.Sound(normalized_data, snd.sampling_frequency)
    def resample_signal(self, snd: parselmouth.Sound) -> parselmouth.Sound:
        """Resamples the audio to the target sample rate using Parselmouth's high-quality resampler."""
        if snd.sampling_frequency != self.target_sr:
            snd = snd.resample(self.target_sr, 50) # 50 is the precision/depth of the Praat resampler
        return snd
    def detect_voice_activity(self, snd: parselmouth.Sound, threshold_db: float = -40.0) -> Tuple[parselmouth.Sound, Dict[str, float]]:
        """
        Energy-based Voice Activity Detection (VAD) and Silence Removal.
        Uses Praat's Intensity contour to robustly detect phonation.
        """
        intensity = snd.to_intensity(minimum_pitch=20.0) # Low minimum pitch to capture deep subharmonics
        intensity_vals = intensity.values[0]
        max_intensity = np.max(intensity_vals)
        # Find frames above threshold
        active_frames = np.where(intensity_vals > (max_intensity + threshold_db))[0]
        if len(active_frames) == 0:
            return snd, {"vad_status": "failed", "active_duration": 0.0}
        start_time = intensity.get_time_from_frame_number(active_frames[0] + 1)
        end_time = intensity.get_time_from_frame_number(active_frames[-1] + 1)
        # Extract active part
        trimmed_snd = snd.extract_part(from_time=start_time, to_time=end_time, preserve_times=False)
        stats = {
            "vad_status": "success",
            "start_time": start_time,
            "end_time": end_time,
            "active_duration": end_time - start_time
        }
        return trimmed_snd, stats
    def run_diagnostics(self, original: parselmouth.Sound, processed: parselmouth.Sound) -> Dict[str, Any]:
        """Generates a comprehensive diagnostic report of the signal quality."""
        orig_data = original.values[0]
        proc_data = processed.values[0]
        # Clipping Detection
        clipping_threshold = 0.99
        clipped_samples = np.sum(np.abs(orig_data) >= clipping_threshold)
        clipping_ratio = float(clipped_samples / len(orig_data))
        # Dynamic Range
        rms_orig = np.sqrt(np.mean(orig_data**2))
        peak_orig = np.max(np.abs(orig_data))
        crest_factor = float(peak_orig / (rms_orig + 1e-10))
        diagnostics = {
            "Input_Quality_Report": {
                "original_sr": original.sampling_frequency,
                "duration_seconds": original.get_total_duration(),
            },
            "Dynamic_Range_Report": {
                "rms_level": float(rms_orig),
                "peak_level": float(peak_orig),
                "crest_factor": crest_factor,
                "dynamic_range_db": float(20 * np.log10(peak_orig / (rms_orig + 1e-10)))
            },
            "Clipping_Detection": {
                "clipped_samples_count": int(clipped_samples),
                "clipping_ratio": clipping_ratio,
                "is_clipping": clipping_ratio > 0.001
            },
            "Sampling_Consistency": {
                "target_sr_achieved": processed.sampling_frequency == self.target_sr,
                "final_sr": processed.sampling_frequency
            },
            "Processing_Validation": {
                "dc_offset_removed": bool(np.abs(np.mean(proc_data)) < 1e-5),
                "peak_normalized_to": float(np.max(np.abs(proc_data)))
            }
        }
        return diagnostics
    def process_pipeline(self, file_path: str) -> Tuple[parselmouth.Sound, Dict[str, Any]]:
        """Executes the full Phase 1 Pipeline sequentially."""
        # 1. Load & Validate
        snd_raw = self.load_and_validate(file_path)
        # 2. Resample
        snd = self.resample_signal(snd_raw)
        # 3. DC Removal
        snd = self.remove_dc_offset(snd)
        # 4. Pre-emphasis (Optional: can be bypassed if low-freq fidelity is absolute priority)
        # snd = self.apply_pre_emphasis(snd)
        # 5. VAD & Silence Detection
        snd, vad_stats = self.detect_voice_activity(snd)
        # 6. Normalization
        snd_final = self.normalize_signal(snd)
        # 7. Diagnostics
        report = self.run_diagnostics(snd_raw, snd_final)
        report["VAD_Stats"] = vad_stats
        return snd_final, report
# Example of instantiation for later modules to hook into:
# engine = SignalAcquisitionEngine()
# processed_signal, diagnostics_report = engine.process_pipeline("vocal_sample.wav")
import numpy as np
import scipy.signal as signal
from scipy.fft import fft, ifft, fftfreq
import pywt
import torch
from typing import Dict, Tuple, Optional, Any, Union
import warnings

warnings.filterwarnings('ignore')

class TimeFrequencyEngine:
    """
    Research-grade Time-Frequency Analysis Module.
    Engineered to isolate complex multi-source phonation dynamics
    and prepare rigorous spectral representations for physical modeling.
    """

    def __init__(self, sample_rate: float = 44100.0, frame_size: int = 2048, hop_size: int = 512):
        self.sr = float(sample_rate)
        self.frame_size = int(frame_size)
        self.hop_size = int(hop_size)

    # ==========================================
    # Time Series Pre-FFT Construction
    # ==========================================

    def generate_sliding_windows(self, audio_data: np.ndarray) -> np.ndarray:
        """
        Frame Generator & Sliding Window:
        Splits 1D audio signal into overlapping 2D frames.
        """
        num_frames = 1 + int((len(audio_data) - self.frame_size) / self.hop_size)
        # Using stride tricks for memory-efficient frame generation
        shape = (num_frames, self.frame_size)
        strides = (audio_data.strides[0] * self.hop_size, audio_data.strides[0])
        frames = np.lib.stride_tricks.as_strided(audio_data, shape=shape, strides=strides)
        return np.copy(frames) # Return copy to avoid memory overlap issues

    def generate_time_indices(self, num_frames: int) -> Tuple[np.ndarray, np.ndarray]:
        """
        Time Index & Frame Index Generator:
        Returns precise timestamps for the center of each frame.
        """
        frame_indices = np.arange(num_frames)
        # Calculate time at the center of the frame
        timestamps = (frame_indices * self.hop_size + (self.frame_size / 2)) / self.sr
        return frame_indices, timestamps

    def apply_window_function(self, frames: np.ndarray, window_type: str = 'hann') -> np.ndarray:
        """Applies mathematical windowing to reduce spectral leakage."""
        if window_type == 'hann':
            win = signal.windows.hann(self.frame_size)
        elif window_type == 'hamming':
            win = signal.windows.hamming(self.frame_size)
        elif window_type == 'blackman':
            win = signal.windows.blackman(self.frame_size)
        else:
            win = np.ones(self.frame_size)

        return frames * win

    # ==========================================
    # Spectral Transforms
    # ==========================================

    def compute_fft(self, frames_windowed: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Computes Fast Fourier Transform for all frames."""
        spectra = fft(frames_windowed, axis=1)
        freqs = fftfreq(self.frame_size, 1 / self.sr)

        # Return only positive frequencies
        pos_mask = freqs >= 0
        return freqs[pos_mask], spectra[:, pos_mask]

    def compute_stft(self, audio_data: np.ndarray, window_type: str = 'hann') -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Standard Short-Time Fourier Transform using SciPy."""
        f, t, Zxx = signal.stft(audio_data, fs=self.sr, window=window_type,
                                nperseg=self.frame_size, noverlap=(self.frame_size - self.hop_size))
        return f, t, Zxx

    def compute_welch_psd(self, audio_data: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Welch's method for Power Spectral Density estimation."""
        f, Pxx = signal.welch(audio_data, fs=self.sr, nperseg=self.frame_size,
                              noverlap=(self.frame_size - self.hop_size))
        return f, Pxx

    def compute_wavelet(self, audio_data: np.ndarray, wavelet: str = 'cmor1.5-1.0', num_scales: int = 128) -> Tuple[np.ndarray, np.ndarray]:
        """
        Continuous Wavelet Transform (CWT).
        Critical for analyzing non-stationary subharmonics and oscillator beating.
        """
        scales = np.geomspace(1, 1024, num=num_scales)
        coefficients, frequencies = pywt.cwt(audio_data, scales, wavelet, sampling_period=1/self.sr)
        return frequencies, coefficients

    def compute_cepstrum(self, power_spectrum: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Computes the Real Cepstrum.
        Essential for separating the vocal source (oscillators) from the vocal tract filter.
        """
        log_spectrum = np.log(np.maximum(power_spectrum, 1e-10))
        cepstrum = np.real(ifft(log_spectrum, axis=1))
        quefrency = np.arange(cepstrum.shape[1]) / self.sr
        return quefrency, cepstrum

    def extract_spectral_envelope(self, log_spectrum: np.ndarray, lifter_cutoff: int = 30) -> np.ndarray:
        """
        Extracts the spectral envelope using Cepstral Liftering.
        Smooths out the harmonics to reveal the formant structure.
        """
        # Forward inverse-FFT to cepstral domain
        cepstrum = np.real(ifft(log_spectrum, axis=1))

        # Apply low-quefrency lifter (keep only the broad envelope)
        lifter = np.zeros_like(cepstrum)
        lifter[:, :lifter_cutoff] = 1
        lifter[:, -lifter_cutoff:] = 1 # Keep symmetric part

        liftered_cepstrum = cepstrum * lifter

        # Back to spectral domain
        envelope = np.real(fft(liftered_cepstrum, axis=1))
        return envelope

    def export_to_tensor(self, matrix: np.ndarray) -> torch.Tensor:
        """Converts numpy structures to PyTorch tensors for physical/AI modeling."""
        return torch.from_numpy(matrix).float()

    # ==========================================
    # Full Phase 2 Pipeline Execution
    # ==========================================

    def process_time_frequency(self, audio_data: np.ndarray) -> Dict[str, Any]:
        """Executes the complete Time-Frequency analysis pipeline."""

        # 1. Time Series Construction
        frames = self.generate_sliding_windows(audio_data)
        frame_idx, timestamps = self.generate_time_indices(frames.shape[0])
        windowed_frames = self.apply_window_function(frames, 'hann')

        # 2. Spectral Transforms
        freqs, stft_complex = self.compute_fft(windowed_frames)
        power_spec = np.abs(stft_complex) ** 2

        # 3. Cepstral & Envelope Analysis
        quefrency, cepstrum = self.compute_cepstrum(power_spec)
        envelope = self.extract_spectral_envelope(np.log(np.maximum(power_spec, 1e-10)))

        # 4. Advanced (Wavelet & Welch) on a subset to save memory if needed
        # We run Welch on the full signal for a global noise profile
        welch_f, welch_p = self.compute_welch_psd(audio_data)

        results = {
            "Time_Series": {
                "timestamps": timestamps,
                "frame_indices": frame_idx,
                "windowed_frames": windowed_frames
            },
            "Frequency_Domain": {
                "frequencies": freqs,
                "stft_magnitude": np.abs(stft_complex),
                "stft_phase": np.angle(stft_complex)
            },
            "Advanced_Analysis": {
                "welch_psd": welch_p,
                "cepstrum": cepstrum,
                "quefrency": quefrency,
                "spectral_envelope": envelope
            }
        }
        return results

# Example Usage:
# tf_engine = TimeFrequencyEngine(sample_rate=44100, frame_size=2048, hop_size=512)
# tf_results = tf_engine.process_time_frequency(processed_audio_numpy_array)
# pytorch_stft = tf_engine.export_to_tensor(tf_results["Frequency_Domain"]["stft_magnitude"])
import numpy as np
import scipy.signal as signal
from typing import Dict, List, Tuple, Any
import warnings

warnings.filterwarnings('ignore')

class SpectralPeakEngine:
    """
    Research-grade Spectral Peak Extraction Engine.
    Detects and refines all spectral peaks without harmonic bias.
    """

    def __init__(self, prominence_threshold: float = 0.05, distance_hz: float = 5.0):
        self.prominence_threshold = prominence_threshold
        self.distance_hz = distance_hz

    def detect_raw_peaks(self, power_spectrum: np.ndarray, freqs: np.ndarray) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        """Identifies raw peaks based on topographical prominence."""
        df = freqs[1] - freqs[0]
        min_distance_bins = max(1, int(self.distance_hz / df))

        peak_indices, properties = signal.find_peaks(
            power_spectrum,
            prominence=self.prominence_threshold,
            distance=min_distance_bins,
            width=1 # Captures peak width for stability analysis
        )
        return peak_indices, properties

    def refine_peaks_parabolic(self, power_spectrum: np.ndarray, freqs: np.ndarray, peak_indices: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Sub-bin precision refinement using Parabolic Interpolation.
        Critical for exact difference-frequency tracking.
        """
        refined_freqs = np.zeros(len(peak_indices))
        refined_amps = np.zeros(len(peak_indices))

        for i, idx in enumerate(peak_indices):
            if 0 < idx < len(power_spectrum) - 1:
                alpha = np.log10(power_spectrum[idx - 1] + 1e-10)
                beta = np.log10(power_spectrum[idx] + 1e-10)
                gamma = np.log10(power_spectrum[idx + 1] + 1e-10)

                # Parabolic interpolation formula
                p = 0.5 * (alpha - gamma) / (alpha - 2 * beta + gamma + 1e-10)

                refined_freqs[i] = freqs[idx] + p * (freqs[1] - freqs[0])
                refined_amps[i] = power_spectrum[idx] # Base amplitude
            else:
                refined_freqs[i] = freqs[idx]
                refined_amps[i] = power_spectrum[idx]

        return refined_freqs, refined_amps

    def calculate_peak_confidence(self, properties: Dict[str, np.ndarray]) -> np.ndarray:
        """Calculates a normalized confidence score based on prominence and sharpness (width)."""
        prominences = properties['prominences']
        widths = properties['widths']

        # High prominence and narrow width = High Confidence (True physical oscillator)
        confidence = (prominences / (np.max(prominences) + 1e-10)) * (1.0 / (widths + 1e-10))
        return confidence / (np.max(confidence) + 1e-10)

    def process_frame_peaks(self, power_spectrum: np.ndarray, freqs: np.ndarray) -> List[Dict[str, float]]:
        """Executes Phase 3 pipeline for a single time frame."""
        peak_indices, properties = self.detect_raw_peaks(power_spectrum, freqs)

        if len(peak_indices) == 0:
            return []

        exact_freqs, exact_amps = self.refine_peaks_parabolic(power_spectrum, freqs, peak_indices)
        confidence_scores = self.calculate_peak_confidence(properties)

        frame_peaks = []
        for i in range(len(exact_freqs)):
            frame_peaks.append({
                "frequency": float(exact_freqs[i]),
                "amplitude": float(exact_amps[i]),
                "confidence": float(confidence_scores[i]),
                "width": float(properties['widths'][i])
            })

        # Sort by amplitude (descending)
        return sorted(frame_peaks, key=lambda x: x['amplitude'], reverse=True)
class CandidateExtractionEngine:
    """
    Research-grade Oscillator Candidate Extraction Engine.
    Replaces traditional F0 detection with Multi-Source and Intermodulation analysis.
    """

    def __init__(self, tolerance_hz: float = 3.0):
        self.tol = tolerance_hz # Frequency matching tolerance

    def _is_harmonic(self, target_f: float, base_f: float) -> bool:
        """Checks if a frequency is an integer multiple of a base frequency."""
        if base_f < 1e-3: return False
        ratio = target_f / base_f
        return abs(ratio - round(ratio)) < (self.tol / base_f)

    def _is_difference_frequency(self, target_f: float, f1: float, f2: float) -> bool:
        """
        Checks for parametric coupling (f1 ± f2) or (n*f1 ± m*f2).
        Parametric coupling produces difference frequencies when system parameters
        are modulated by oscillatory signals.
        """
        diff = abs(f1 - f2)
        return abs(target_f - diff) < self.tol

    def classify_candidates(self, peaks: List[Dict[str, float]]) -> Dict[str, List[Dict[str, float]]]:
        """
        Evaluates peaks to extract mechanical source candidates and nonlinear artifacts.
        """
        candidates = {
            "True_Fold_Candidate": [],
            "Ventricular_Candidate": [],
            "Harmonic_Candidate": [],
            "Difference_Frequency_Candidate": [],
            "Noise_Candidate": []
        }

        if not peaks:
            return candidates

        # 1. Hypothesize Base Oscillators
        # The highest confidence peaks are generally the primary mechanical drivers.
        # Assuming top 2 distinct peaks are the two independent sources (Vocal & Ventricular)
        sorted_by_conf = sorted(peaks, key=lambda x: x['confidence'], reverse=True)

        f1_candidate = sorted_by_conf[0]
        candidates["True_Fold_Candidate"].append(f1_candidate)

        f2_candidate = None
        for p in sorted_by_conf[1:]:
            # Find the next strong peak that is NOT a harmonic of f1
            if not self._is_harmonic(p['frequency'], f1_candidate['frequency']):
                f2_candidate = p
                candidates["Ventricular_Candidate"].append(f2_candidate)
                break

        # 2. Classify the rest of the ecosystem
        for p in peaks:
            # Skip if already classified as base sources
            if p in candidates["True_Fold_Candidate"] or p in candidates["Ventricular_Candidate"]:
                continue

            f_val = p['frequency']
            is_classified = False

            # Check Harmonicity
            if self._is_harmonic(f_val, f1_candidate['frequency']) or \
               (f2_candidate and self._is_harmonic(f_val, f2_candidate['frequency'])):
                candidates["Harmonic_Candidate"].append(p)
                is_classified = True

            # Check Difference Frequencies (Intermodulation)
            if f2_candidate and not is_classified:
                if self._is_difference_frequency(f_val, f1_candidate['frequency'], f2_candidate['frequency']):
                    candidates["Difference_Frequency_Candidate"].append(p)
                    is_classified = True

            # Noise / Unexplained Non-linearities
            if not is_classified:
                if p['confidence'] < 0.2:
                    candidates["Noise_Candidate"].append(p)
                else:
                    # High confidence but unexplained -> Could be higher-order bifurcation
                    candidates["Difference_Frequency_Candidate"].append(p)

        return candidates

    def build_time_series_tracker(self, frames_candidates: List[Dict[str, List[Dict[str, float]]]]) -> Dict[str, np.ndarray]:
        """
        Tracks the candidates over time to observe entrainment, gliding, and bifurcations.
        Returns continuous trajectories suitable for PyTorch ingestion.
        """
        # Logic to stitch frame-by-frame candidates into continuous time-series arrays
        # (Implementation applies nearest-neighbor tracking across time frames)
        pass
import numpy as np
from typing import Dict, List, Optional, Tuple, Any
import warnings

warnings.filterwarnings('ignore')

class HarmonicEngine:
    """
    Research-grade Harmonic Analysis Engine.
    Constructs, tracks, and evaluates harmonic series for specific oscillator candidates,
    accounting for inharmonicity and multi-source spectral overlap.
    """

    def __init__(self, base_tolerance_hz: float = 3.0, inharmonicity_factor: float = 0.01):
        # Base tolerance for H1
        self.base_tolerance = base_tolerance_hz
        # Allows tolerance to widen slightly for higher harmonics due to tissue stiffness
        self.inharmonicity_factor = inharmonicity_factor

    def calculate_harmonic_number(self, target_f: float, base_f: float) -> Tuple[int, float]:
        """
        Determines the theoretical harmonic number (n) and the exact frequency deviation.
        """
        if base_f < 1e-3:
            return 0, 0.0

        ratio = target_f / base_f
        n = int(round(ratio))

        expected_f = n * base_f
        deviation_hz = abs(target_f - expected_f)

        return n, deviation_hz

    def calculate_dynamic_tolerance(self, harmonic_number: int) -> float:
        """Widens the search window for higher harmonics to account for natural physical inharmonicity."""
        return self.base_tolerance + (self.inharmonicity_factor * harmonic_number * self.base_tolerance)

    def calculate_harmonic_confidence(self, peak_confidence: float, deviation_hz: float, tolerance: float) -> float:
        """
        Fuses the raw peak confidence (from Phase 3) with its mathematical harmonic alignment.
        """
        # Penalty increases as deviation approaches the tolerance limit
        alignment_score = max(0.0, 1.0 - (deviation_hz / (tolerance + 1e-10)))
        return float(peak_confidence * alignment_score)

    def assess_harmonic_stability(self, amplitude_history: List[float], freq_history: List[float]) -> Dict[str, float]:
        """
        Evaluates the temporal stability of a specific harmonic across previous frames.
        """
        if len(amplitude_history) < 2:
            return {"amp_stability": 1.0, "freq_stability": 1.0}

        amp_variance = np.var(amplitude_history)
        freq_variance = np.var(freq_history)

        # Inverse variance mapping to a 0-1 score
        amp_stability = 1.0 / (1.0 + float(amp_variance))
        freq_stability = 1.0 / (1.0 + float(freq_variance))

        return {"amp_stability": amp_stability, "freq_stability": freq_stability}

    def build_harmonic_series(self, base_candidate: Dict[str, float], all_peaks: List[Dict[str, float]], max_harmonics: int = 50) -> Dict[str, Any]:
        """
        Builds the complete harmonic series (H1, H2, H3...) for a given mechanical source.
        """
        base_f = base_candidate['frequency']
        harmonic_series = {
            "base_frequency": base_f,
            "harmonics": {}, # Key: Harmonic Number (int), Value: Harmonic Data
            "spectral_centroid": 0.0,
            "energy_distribution": []
        }

        total_energy = 0.0
        weighted_freq_sum = 0.0

        for peak in all_peaks:
            n, deviation = self.calculate_harmonic_number(peak['frequency'], base_f)

            # Skip subharmonics or excessively high harmonics for this series
            if n < 1 or n > max_harmonics:
                continue

            dynamic_tol = self.calculate_dynamic_tolerance(n)

            if deviation <= dynamic_tol:
                h_confidence = self.calculate_harmonic_confidence(peak['confidence'], deviation, dynamic_tol)

                harmonic_data = {
                    "harmonic_number": n,
                    "frequency": peak['frequency'],
                    "amplitude": peak['amplitude'],
                    "deviation_hz": deviation,
                    "confidence": h_confidence,
                    "width": peak.get('width', 0.0)
                }

                # Handle potential overlaps (e.g., if two peaks fall within the tolerance for H_n, pick the one with higher confidence)
                if n in harmonic_series["harmonics"]:
                    if h_confidence > harmonic_series["harmonics"][n]["confidence"]:
                        harmonic_series["harmonics"][n] = harmonic_data
                else:
                    harmonic_series["harmonics"][n] = harmonic_data

        # Calculate derived series metrics (Centroid & Energy)
        for n, h_data in harmonic_series["harmonics"].items():
            amp = h_data["amplitude"]
            total_energy += amp
            weighted_freq_sum += h_data["frequency"] * amp
            harmonic_series["energy_distribution"].append((n, amp))

        if total_energy > 0:
            harmonic_series["spectral_centroid"] = weighted_freq_sum / total_energy

        return harmonic_series

    def track_series_across_frames(self, current_series: Dict[str, Any], previous_series: Dict[str, Any]) -> Dict[str, Any]:
        """
        Links harmonic numbers across time frames to calculate stability and track micro-glides.
        """
        tracked_series = current_series.copy()

        for n, current_h in tracked_series["harmonics"].items():
            if n in previous_series["harmonics"]:
                prev_h = previous_series["harmonics"][n]

                # Mock histories (in a real pipeline, these would be fetched from a state manager/buffer)
                mock_amp_history = [prev_h["amplitude"], current_h["amplitude"]]
                mock_freq_history = [prev_h["frequency"], current_h["frequency"]]

                stability = self.assess_harmonic_stability(mock_amp_history, mock_freq_history)
                current_h["amp_stability"] = stability["amp_stability"]
                current_h["freq_stability"] = stability["freq_stability"]
            else:
                # New harmonic appeared
                current_h["amp_stability"] = 0.0
                current_h["freq_stability"] = 0.0

        return tracked_series
import numpy as np
from typing import Dict, List, Optional, Tuple, Any
import warnings

warnings.filterwarnings('ignore')

class DualOscillatorEngine:
    """
    Research-grade Dual Oscillator Tracking Engine.
    Engineered to model True Vocal Folds (TVF) and Ventricular Folds (VF)
    as independent but dynamically coupled biomechanical systems.
    """

    def __init__(self, entrainment_threshold_hz: float = 10.0):
        # Threshold at which oscillators are considered to be crossing or entraining
        self.entrainment_threshold = entrainment_threshold_hz

        # Internal state memory for continuous time-series tracking
        self.time_series_history = {
            "TVF_trajectory": [],
            "VF_trajectory": [],
            "Crossing_events": [],
            "Energy_exchange": []
        }

        # Last known states to prevent identity swapping during crossings
        self._last_tvf_freq = None
        self._last_vf_freq = None

    def calculate_oscillator_energy(self, harmonic_series: Dict[str, Any]) -> float:
        """
        Integrates the total acoustic energy of a single oscillator
        based on its harmonic series distribution.
        """
        energy = 0.0
        if not harmonic_series or "harmonics" not in harmonic_series:
            return energy

        # Parseval's theorem approximation: Sum of squared amplitudes
        for n, h_data in harmonic_series["harmonics"].items():
            energy += (h_data["amplitude"] ** 2)

        return float(energy)

    def track_oscillator_identity(self, candidates: Dict[str, List[Dict[str, float]]]) -> Tuple[Optional[Dict[str, float]], Optional[Dict[str, float]]]:
        """
        Robustly tracks TVF and VF identities across frames to prevent swapping
        when frequencies cross, using nearest-neighbor trajectory matching.
        """
        tvf_cands = candidates.get("True_Fold_Candidate", [])
        vf_cands = candidates.get("Ventricular_Candidate", [])

        current_tvf = tvf_cands[0] if tvf_cands else None
        current_vf = vf_cands[0] if vf_cands else None

        # Identity preservation logic using historical state
        if current_tvf and current_vf and self._last_tvf_freq and self._last_vf_freq:
            # Calculate distances to last known states
            dist_tvf_to_tvf = abs(current_tvf['frequency'] - self._last_tvf_freq)
            dist_vf_to_vf = abs(current_vf['frequency'] - self._last_vf_freq)

            dist_tvf_to_vf = abs(current_tvf['frequency'] - self._last_vf_freq)
            dist_vf_to_tvf = abs(current_vf['frequency'] - self._last_tvf_freq)

            # If swapped distances are smaller, swap the identities for this frame
            if (dist_tvf_to_vf + dist_vf_to_tvf) < (dist_tvf_to_tvf + dist_vf_to_vf):
                current_tvf, current_vf = current_vf, current_tvf

        # Update last known states
        if current_tvf: self._last_tvf_freq = current_tvf['frequency']
        if current_vf: self._last_vf_freq = current_vf['frequency']

        return current_tvf, current_vf

    def detect_oscillator_crossing(self, tvf_freq: float, vf_freq: float) -> Dict[str, Any]:
        """
        Detects if the natural frequencies of the two oscillators are approaching
        each other, leading to energy exchange, bifurcations, or entrainment.
        """
        distance = abs(tvf_freq - vf_freq)
        is_crossing = distance < self.entrainment_threshold

        crossing_data = {
            "is_crossing": is_crossing,
            "frequency_distance": float(distance),
            "entrainment_risk": 1.0 - min(1.0, distance / (self.entrainment_threshold * 2 + 1e-10))
        }
        return crossing_data

    def evaluate_oscillator_stability(self, current_freq: float, history: List[float]) -> float:
        """Calculates instantaneous stability (jitter equivalent) for an oscillator."""
        if len(history) < 3:
            return 1.0 # Max stability if no history

        recent_history = history[-3:] # Look at last 3 frames
        mean_freq = np.mean(recent_history)
        variance = np.var(recent_history)

        # Stability score between 0 (chaotic) and 1 (perfectly stable)
        stability = 1.0 / (1.0 + float(variance / (mean_freq + 1e-10)))
        return stability

    def process_dual_oscillators(self, timestamp: float, candidates: Dict[str, List[Dict[str, float]]],
                                 tvf_harmonic_series: Dict[str, Any],
                                 vf_harmonic_series: Dict[str, Any]) -> Dict[str, Any]:
        """
        Executes Phase 6 pipeline: Assigns identities, calculates physics,
        and updates the continuous time-series state.
        """
        tvf_peak, vf_peak = self.track_oscillator_identity(candidates)

        state = {
            "timestamp": timestamp,
            "TVF_State": None,
            "VF_State": None,
            "Interaction": None
        }

        # 1. Update TVF State
        if tvf_peak:
            tvf_energy = self.calculate_oscillator_energy(tvf_harmonic_series)
            tvf_stability = self.evaluate_oscillator_stability(tvf_peak['frequency'], self.time_series_history["TVF_trajectory"])

            state["TVF_State"] = {
                "frequency": float(tvf_peak['frequency']),
                "amplitude": float(tvf_peak['amplitude']),
                "confidence": float(tvf_peak['confidence']),
                "energy": tvf_energy,
                "stability": tvf_stability
            }
            self.time_series_history["TVF_trajectory"].append(tvf_peak['frequency'])
        else:
            self.time_series_history["TVF_trajectory"].append(np.nan)

        # 2. Update VF State
        if vf_peak:
            vf_energy = self.calculate_oscillator_energy(vf_harmonic_series)
            vf_stability = self.evaluate_oscillator_stability(vf_peak['frequency'], self.time_series_history["VF_trajectory"])

            state["VF_State"] = {
                "frequency": float(vf_peak['frequency']),
                "amplitude": float(vf_peak['amplitude']),
                "confidence": float(vf_peak['confidence']),
                "energy": vf_energy,
                "stability": vf_stability
            }
            self.time_series_history["VF_trajectory"].append(vf_peak['frequency'])
        else:
            self.time_series_history["VF_trajectory"].append(np.nan)

        # 3. Analyze Coupling & Crossing (Interaction)
        if tvf_peak and vf_peak:
            interaction = self.detect_oscillator_crossing(tvf_peak['frequency'], vf_peak['frequency'])

            # Energy ratio (which oscillator is dominating?)
            total_energy = state["TVF_State"]["energy"] + state["VF_State"]["energy"] + 1e-10
            interaction["tvf_energy_dominance"] = state["TVF_State"]["energy"] / total_energy
            interaction["vf_energy_dominance"] = state["VF_State"]["energy"] / total_energy

            state["Interaction"] = interaction
            self.time_series_history["Crossing_events"].append(interaction["is_crossing"])
            self.time_series_history["Energy_exchange"].append((interaction["tvf_energy_dominance"], interaction["vf_energy_dominance"]))
        else:
            self.time_series_history["Crossing_events"].append(False)
            self.time_series_history["Energy_exchange"].append((0.0, 0.0))

        return state

    def export_time_series_tensors(self) -> Dict[str, np.ndarray]:
        """
        Exports the entire recorded timeline as continuous numpy arrays,
        ready for PyTorch physics engine ingestion.
        """
        return {
            "TVF_F0_Trajectory": np.array(self.time_series_history["TVF_trajectory"]),
            "VF_F0_Trajectory": np.array(self.time_series_history["VF_trajectory"]),
            "Crossing_Flags": np.array(self.time_series_history["Crossing_events"], dtype=bool)
        }
import numpy as np
from typing import Dict, List, Any
import warnings

warnings.filterwarnings('ignore')

class SignalQualityEngine:
    """
    Research-grade Signal Quality & Reliability Engine (Phase 9).
    Quantifies noise floors, signal-to-noise ratios (SNR), signal quality indices (SQI),
    drifts, and overall signal reliability for multi-source phonation.
    """

    def __init__(self, sample_rate: int = 44100):
        self.sample_rate = sample_rate

    def calculate_noise_floor_and_snr(self, power_spectrum: np.ndarray, peak_frequencies: List[float], power_threshold: float = 0.01) -> Dict[str, float]:
        """
        Calculates the local noise floor and Signal-to-Noise Ratio (SNR)
        by excluding identified spectral peaks.
        """
        if len(power_spectrum) == 0:
            return {"noise_floor": 0.0, "snr_db": 0.0}

        # Create a mask to exclude bins near known active peaks
        mask = np.ones(len(power_spectrum), dtype=bool)
        freqs = np.fft.rfftfreq(len(power_spectrum) * 2 - 2, 1 / self.sample_rate)

        df = freqs[1] - freqs[0] if len(freqs) > 1 else 1.0
        for f in peak_frequencies:
            idx = int(f / df)
            # Mask out a small window around each peak (e.g., ±15 Hz)
            window_bins = max(1, int(15.0 / df))
            start = max(0, idx - window_bins)
            end = min(len(mask), idx + window_bins)
            mask[start:end] = False

        noise_bins = power_spectrum[mask]
        if len(noise_bins) == 0:
            noise_floor = float(np.min(power_spectrum))
        else:
            # Robust noise floor estimation using median energy of non-peak bins
            noise_floor = float(np.median(noise_bins))

        # Peak signal energy estimation
        peak_energies = [p for p in power_spectrum if p > noise_floor]
        signal_energy = float(np.mean(peak_energies)) if peak_energies else noise_floor

        # Calculate SNR in decibels (dB)
        snr_db = 10.0 * np.log10((signal_energy + 1e-10) / (noise_floor + 1e-10))

        return {
            "noise_floor": noise_floor,
            "snr_db": float(snr_db)
        }

    def calculate_sqi(self, snr_db: float, harmonic_stability: float, baseline_drift: float) -> float:
        """
        Calculates a composite Signal Quality Index (SQI) on a scale of 0.0 to 1.0
        to determine if the frame is reliable for deep-tech synthesis.
        """
        # Normalize SNR (assume 0 dB is bad, 40+ dB is excellent)
        norm_snr = min(1.0, max(0.0, snr_db / 40.0))

        # Penalize for baseline drift and reward harmonic stability
        sqi = (0.4 * norm_snr) + (0.4 * harmonic_stability) + (0.2 * max(0.0, 1.0 - baseline_drift))
        return float(max(0.0, min(1.0, sqi)))

    def evaluate_drifts(self, current_frame_features: Dict[str, float], history_frames: List[Dict[str, float]]) -> Dict[str, float]:
        """
        Tracks frequency, amplitude, and baseline shifts over continuous sliding windows.
        """
        if not history_frames:
            return {"frequency_drift": 0.0, "amplitude_drift": 0.0, "baseline_drift": 0.0}

        prev_f = history_frames[-1].get("frequency", current_frame_features.get("frequency", 0.0))
        prev_amp = history_frames[-1].get("amplitude", current_frame_features.get("amplitude", 0.0))
        prev_base = history_frames[-1].get("baseline_energy", current_frame_features.get("baseline_energy", 0.0))

        curr_f = current_frame_features.get("frequency", 0.0)
        curr_amp = current_frame_features.get("amplitude", 0.0)
        curr_base = current_frame_features.get("baseline_energy", 0.0)

        # Instantaneous drift magnitudes
        freq_drift = abs(curr_f - prev_f)
        amp_drift = abs(curr_amp - prev_amp)
        baseline_drift = abs(curr_base - prev_base)

        return {
            "frequency_drift": float(freq_drift),
            "amplitude_drift": float(amp_drift),
            "baseline_drift": float(baseline_drift)
        }

    def assess_signal_reliability(self, sqi: float, snr_db: float, freq_drift: float) -> Tuple[bool, str]:
        """
        Defines binary reliability flag and diagnostic category for the current frame.
        """
        # Strict thresholds for research-grade processing
        if sqi < 0.35 or snr_db < 6.0 or freq_drift > 50.0:
            return False, "CORRUPT_OR_UNSTABLE"
        elif sqi < 0.65:
            return True, "MARGINAL_RELIABILITY"
        else:
            return True, "PRISTINE_RESEARCH_GRADE"

    def process_signal_quality(self, power_spectrum: np.ndarray, peak_list: List[Dict[str, float]],
                               current_features: Dict[str, float], history: List[Dict[str, float]]) -> Dict[str, Any]:
        """
        Executes Phase 9 pipeline: Computes noise metrics, SQI, drifts, and overall reliability.
        """
        peak_freqs = [p['frequency'] for p in peak_list]

        # 1. Noise Floor & SNR
        noise_metrics = self.calculate_noise_floor_and_snr(power_spectrum, peak_freqs)

        # 2. Drifts evaluation
        drifts = self.evaluate_drifts(current_features, history)

        # Approximate harmonic stability from current features or history
        harmonic_stability = current_features.get("harmonic_stability", 0.8)

        # 3. SQI Calculation
        sqi = self.calculate_sqi(noise_metrics["snr_db"], harmonic_stability, drifts["baseline_drift"])

        # 4. Reliability Decision
        is_reliable, status_tag = self.assess_signal_reliability(sqi, noise_metrics["snr_db"], drifts["frequency_drift"])

        quality_report = {
            "noise_floor": noise_metrics["noise_floor"],
            "snr_db": noise_metrics["snr_db"],
            "sqi": sqi,
            "harmonic_stability": harmonic_stability,
            "frequency_drift": drifts["frequency_drift"],
            "amplitude_drift": drifts["amplitude_drift"],
            "baseline_drift": drifts["baseline_drift"],
            "is_reliable": is_reliable,
            "reliability_status": status_tag
        }

        return quality_report
